In [1]:
import numpy as np
import qiskit_metal as metal
import qiskit_metal as qm
from qiskit_metal import MetalGUI
from qiskit_metal import designs, draw
from qiskit_metal import Dict, open_docs
import pandas as pd

%metal_heading Welcome to Qiskit Metal!


In [2]:
#Speed of Light
c = 3*10**8 

In [3]:
design = designs.DesignPlanar()
gui = MetalGUI(design)

In [4]:
design.overwrite_enabled =  True

design.chips.main.size.size_x = '6.4mm'
design.chips.main.size.size_y = '6.4 mm'
design.chips.main.size.size_z = '-270um'
design.chips.main.size.sample_holder_top = '50um'
design.chips.main.size.sample_holder_bottom = '800um'

In [5]:
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '6.4mm',
  'size_y': '6.4 mm',
  'size_z': '-270um',
  'sample_holder_top': '50um',
  'sample_holder_bottom': '800um'}}

In [6]:
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven

In [7]:
port_1 = LaunchpadWirebondDriven(design,'port_1', options = dict(pos_x = '-2.496mm',
                                                                 pos_y = '-0mm',
                                                                 orientation = '0',
                                                                 pad_width = '350um',
                                                                 pad_height = '129um',
                                                                 pad_gap = '150um',
                                                                 trace_width = '10um',
                                                                 trace_gap = '6um',
                                                                 lead_length = '9um',
                                                                 taper_height = '350um'))

In [8]:
port_2 = LaunchpadWirebondDriven(design,'port_2',options = dict(pos_x = '2.496mm',
                                                                pos_y = '-0mm',
                                                                orientation = '180',
                                                                pad_width = '350um',
                                                                pad_height = '129um',
                                                                taper_height = '350um',
                                                                lead_length = '9um',
                                                                trace_width = '10um',
                                                                trace_gap = '6um',
                                                                pad_gap = '150um'))

In [9]:
gui.rebuild()

In [10]:
gui.autoscale()

In [11]:
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

In [12]:
print(design.components['port_1'].pins.keys())

dict_keys(['tie', 'in'])


In [13]:
bus_1 = RouteStraight(design,
                      'bus_1',
                      options = dict(hfss_wire_bonds = True,
                                     pin_inputs = dict(start_pin=dict(component='port_1',pin='tie'),
                                                        end_pin=dict(component='port_2',pin='tie')),
                                    trace_width = '10um',
                                    trace_gap = '6um',))

gui.rebuild()
gui.autoscale()

In [14]:
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

In [15]:
coupl_1 = CoupledLineTee(design,'coupl_1', options = dict(pos_x='-.55mm',
                                                          pos_y = '0',
                                                          orientation = '180',
                                                          coupling_length = '650um',
                                                          down_length = '100um',
                                                          fillet ='50um',
                                                          coupling_space ='4um',
                                                          open_termination = True))

gui.rebuild()
gui.autoscale()

In [16]:
from qiskit_metal.analyses.em.cpw_calculations import guided_wavelength

In [17]:
def find_resonator_length (frequency,line_width,line_gap,N):
    [lambdaG, etfSqrt, q] = guided_wavelength(frequency,line_width,
                                              line_gap, substrate_thickness=270*(10**-6),
                                              film_thickness=200*(10**-9))
    return str(lambdaG/N*10**3)+' mm'

In [18]:
#Half Wavelength Resonator
find_resonator_length(6.5*(10**9),10*(10**-6),6*(10**-6),2)

'9.364582427705939 mm'

### Theortical Frequency Prediction

The resonant frequency of a CPW resonator can be found using the followng equation[1]:
$$
f_0 = \frac{c}{\sqrt{\epsilon_{eff}}}\frac{1}{2l}
$$
where $c$ is the speed of light, $\epsilon_{eff}$ is th effective permitivity of CPW line and $l$ is length of the resonantor with a factor of $2$ as it is a half wavelength resonator which implies $2l = \lambda_0$, the fundamental mode. 

To calculate the the effective permeability the following equation is used[2].
$$
\epsilon_{eff} = \frac{\epsilon_r + 1}{2} + \frac{\epsilon_r - 1}{2} \frac{1}{\sqrt{1+12d/W}}
$$  
For a microstrip line built on top a single substrate.

In [19]:
def find_effective_permeability(e_r,d,W):
    e_e = ((e_r+1)/(2))+((e_r-1)/(2))*(1/(np.sqrt(1+(12*d/W))))
    return e_e

def find_resonator_frequency(e_r,d,W,l):
    e_e = find_effective_permeability(e_r,d,W)

    f0 = (c/(np.sqrt(e_e)))*(1/(2*l))
    return f0


In [20]:
res_f = find_resonator_frequency(11.6,270*(10**-6),10*(10**-6),12*(10**-3))
print(f'{res_f/(10**9)} GHz')

4.867834838432226 GHz


In [21]:
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

In [22]:
open_end = OpenToGround(design, 'open_end',options=dict(pos_x='-0.55mm',
                                                        pos_y='2.25mm',
                                                        orientation='90',
                                                        termination_gap='6um',
                                                        gap='6um',
                                                        width = '10um'))

gui.rebuild()
gui.autoscale()

In [23]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

In [24]:
res_1 = RouteMeander(design, 'res_1', options = dict(
                                                    pin_inputs = dict(start_pin=dict(component = 'coupl_1',
                                                                                     pin = 'second_end'),
                                                                    end_pin=dict(component='open_end',
                                                                                 pin='open')),
                                                    lead =dict(start_straigt = '0um',
                                                               end_straight ='0um'),
                                                    meander = dict(spacing='150um',
                                                                   asymmetry='-300um'),
                                                    hfss_wire_bonds=True,
                                                    total_length='12mm',
                                                    trace_width='10um',
                                                    fillet='50um',
                                                    trace_gap='6um'))

gui.rebuild()
gui.autoscale()

In [25]:
# QDesign registers GDS renderer during init of QDesign.
a_gds = design.renderers.gds
# An alternate way to invoke gds commands without using a_gds:
# design.renderers.gds.export_to_gds()

# Show the options for GDS
a_gds.options

{'short_segments_to_not_fillet': 'True',
 'check_short_segments_by_scaling_fillet': '2.0',
 'gds_unit': 0.001,
 'ground_plane': 'True',
 'negative_mask': {'main': []},
 'fabricate': 'False',
 'corners': 'natural',
 'tolerance': '0.00001',
 'precision': '0.000000001',
 'width_LineString': '10um',
 'path_filename': '../resources/Fake_Junctions.GDS',
 'junction_pad_overlap': '5um',
 'max_points': '199',
 'cheese': {'datatype': '100',
  'shape': '0',
  'cheese_0_x': '25um',
  'cheese_0_y': '25um',
  'cheese_1_radius': '100um',
  'view_in_file': {'main': {1: True}},
  'delta_x': '100um',
  'delta_y': '100um',
  'edge_nocheese': '200um'},
 'no_cheese': {'datatype': '99',
  'buffer': '25um',
  'cap_style': '2',
  'join_style': '2',
  'view_in_file': {'main': {1: True}}},
 'bounding_box_scale_x': '1.2',
 'bounding_box_scale_y': '1.2'}

In [26]:
a_gds.export_to_gds("Single Resonator.gds")

10:59AM 52s WARNING [import_junction_gds_file]: Not able to find file:"../resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"c:\Users\Benji\resources".


1

In [27]:
design.chips['main']

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '6.4mm',
  'size_y': '6.4 mm',
  'size_z': '-270um',
  'sample_holder_top': '50um',
  'sample_holder_bottom': '800um'}}

In [28]:
gui.figure.savefig('shot.png')

the expected resonant frequencu is $f_0 = 4.6385 \mathrm{GHz}$

klayout to qiskit metal

cursor

[1] Coplanar Waveguide Resonators for Circuit Quantum Electrodynamics : Goppl
[2] Microwave Engineering : Pozar